In [ ]:
!pip install transformers[torch] sentence-transformers datasets
!pip install -U accelerate
!pip install -U transformers
!pip install -U evaluate
!pip install -U peft
!pip install seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
API_KEY = "insert_key_here"
os.environ["WANDB_API_KEY"] = API_KEY
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
import torch
import transformers
import nltk
from nltk.corpus import wordnet,subjectivity,stopwords
from nltk.tag.perceptron import PerceptronTagger
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize


In [ ]:
nltk.download("subjectivity")
nltk.download('stopwords')

[nltk_data] Downloading package subjectivity to /root/nltk_data...
[nltk_data]   Package subjectivity is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
def nltk_pos_tagger(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return None

In [ ]:
stop_words = set(stopwords.words("english"))
def tokenize_samples(samples):
    tokenized_samples = []
    for sample in samples:
        tokens = []
        # Split text into sentences
        sentences = sent_tokenize(sample)
        for sent in sentences:
            # Tokenize each sentence into words
            words = word_tokenize(sent)
            for word in words:
                # Filter out stopwords and unwanted tokens
                if '\n' in word or "\t" in word or "--" in word or "*" in word or word.lower() in stop_words:
                    continue
                if word.strip():
                    # Process the token and add to list
                    tokens.append(word.replace('"', "'").strip().lower())
        tokenized_samples.append(tokens)

    return tokenized_samples

In [ ]:
def prepare_data_of_subj_obj():
    print(subjectivity.categories())
# print(subjectivity.sents(categories="subj"))
    subjective_sentences = subjectivity.sents(categories="subj")
    objective_sentences = subjectivity.sents(categories="obj")

    data_with_labels = []
    assert(len(subjective_sentences) == len(objective_sentences))
    for i in range(len(subjective_sentences)):
        obj_sentence = objective_sentences[i]
        subj_sentence = subjective_sentences[i]
        data_with_labels.append({"label":"obj","sentence":obj_sentence})
        data_with_labels.append({"label":"subj","sentence":subj_sentence})

    # print(upenn_tagset()) -> Μέρη του λόγου σε προτάσεις και σε τι κλήση κλπ βρίσκονται
    lemmatizer = WordNetLemmatizer()
    tagger = PerceptronTagger()
    docs = []
    labels = []
    for doc in tqdm(data_with_labels):
        sentence = " ".join(doc["sentence"])
        label = doc["label"]
        document = re.sub(r'\W', ' ', str(sentence))

        document = re.sub(r'\s+[a-zA-Z]\s+', ' ', document)

        document = re.sub(r'\s+', ' ', document, flags=re.I)

        document = document.lower()

        document = document.split()

        doc_pos = [x[1] for x in tagger.tag(document)]

        document = \
            [lemmatizer.lemmatize(token, pos=nltk_pos_tagger(pos_tag)) \
                if nltk_pos_tagger(pos_tag) != None else lemmatizer.lemmatize(token) \
            for token, pos_tag in zip(document, doc_pos)]

        document = ' '.join(document)

        docs.append(document)
        value_of_label = 0 if label == "subj" else 1
        labels.append(value_of_label)
        # labels.append(label)
    # print(docs)
    print(len(subjective_sentences),len(objective_sentences))
    return docs,labels

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import DatasetDict,load_dataset,ClassLabel
id2label = {0:"SUBJ",1:"OBJ"}
label2id = {"SUBJ":0,"OBJ":1}

MODEL = "prajjwal1/bert-small"

dataset = load_dataset("tasksource/subjectivity")
# print(dataset['train'][0])
num_classes = 2
class_label = ClassLabel(num_classes=num_classes,names=['SUBJ','OBJ'])
def convert_labels(example):
    example["labels"] = label2id[example["Label"]]
    return example
# Set aside the test split of the dataset to perform a split on it
dataset['train'] = dataset['train'].map(convert_labels)
dataset['train'] = dataset['train'].class_encode_column('labels')

test_dataset = dataset['test'].map(convert_labels)
test_dataset = test_dataset.class_encode_column('labels')

# test_dataset.cast_column("Label",class_label)
# Perform stratified split using the label
stratified_split = test_dataset.train_test_split(
    test_size=0.5,
    stratify_by_column="labels"
)

print(stratified_split['train'][0])
# Extract the new splits
validation_dataset = stratified_split['test']
test_dataset = stratified_split['train']
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': validation_dataset,
    'test': test_dataset,
})

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2, id2label = id2label, label2id = label2id)

# Freeze the base model (DistilRoBERTa)
for param in model.base_model.parameters():
    param.requires_grad = False

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

Map:   0%|          | 0/731 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/731 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/731 [00:00<?, ? examples/s]

{'Sentence': 'This is well-known, and when not denying, the MSM either justifies it or encourages it—see a partial list of our coverage below:  The third bemoans Trump’s attack on progressive prosecutors—many of whom are black—who spend more time on his business dealings than on crime.\n', 'Label': 'SUBJ', 'Solved conflict': False, 'labels': 0}


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-small and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 28764674
Trainable parameters: 1026


In [ ]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-1

In [ ]:
print(dataset)
def preprocess_function(examples):
    return tokenizer(examples["Sentence"], truncation=True)

tokenized_subj = dataset.map(preprocess_function, batched=True)

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Label', 'Solved conflict', 'labels'],
        num_rows: 731
    })
    validation: Dataset({
        features: ['Sentence', 'Label', 'Solved conflict', 'labels'],
        num_rows: 110
    })
    test: Dataset({
        features: ['Sentence', 'Label', 'Solved conflict', 'labels'],
        num_rows: 109
    })
})


Map:   0%|          | 0/731 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/109 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_subj)
print(tokenized_subj['train'][0])

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Label', 'Solved conflict', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 731
    })
    validation: Dataset({
        features: ['Sentence', 'Label', 'Solved conflict', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 110
    })
    test: Dataset({
        features: ['Sentence', 'Label', 'Solved conflict', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 109
    })
})
{'Sentence': 'In the early hours of Thursday morning, the White House announced a tentative agreement between the major rail carriers and the remaining holdout unions, Teamsters’ Brotherhood of Locomotive Engineers and Trainmen (BLET) and the Transportation Division of the International Association of Sheet Metal, Air, Rail, and Transportation Workers (SMART-TD).\n', 'Label': 'OBJ', 'Solved conflict': False, 'input_ids': [101, 1999, 1996, 2220, 2847, 1997, 9432, 2851, 1010, 199

In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

import torch

def compute_metrics(p):
    predictions, labels = p

    # Ensure predictions are converted to a PyTorch tensor
    if not isinstance(predictions, torch.Tensor):
        predictions = torch.tensor(predictions)

    # Convert logits to predicted class indices
    predictions = torch.argmax(predictions, dim=1)
    # Compute metrics
    accuracy = accuracy_metric.compute(predictions=predictions.numpy(), references=labels)
    f1 = f1_metric.compute(predictions=predictions.numpy(), references=labels, average="macro")

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt") # Dynamically pads the sequences within each patch to avoid any shape misalignments

training_args = TrainingArguments(
    output_dir='./txt_cls_frozen_base/',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_subj["train"],
    eval_dataset=tokenized_subj["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-57-f4b89c166270>:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.768483,0.481818,0.325153
2,No log,0.767772,0.481818,0.325153


TrainOutput(global_step=92, training_loss=0.6578150210173234, metrics={'train_runtime': 68.4295, 'train_samples_per_second': 21.365, 'train_steps_per_second': 1.344, 'total_flos': 10544931143424.0, 'train_loss': 0.6578150210173234, 'epoch': 2.0})

In [ ]:
predictions = trainer.predict(test_dataset=tokenized_subj["test"])

# Unpack the results
logits = predictions.predictions
labels = predictions.label_ids
metrics = predictions.metrics

metrics

{'test_loss': 0.7513681054115295,
 'test_accuracy': 0.48623853211009177,
 'test_f1': 0.3271604938271605,
 'test_runtime': 3.4317,
 'test_samples_per_second': 31.763,
 'test_steps_per_second': 2.04}